# 05. LiRA and the most-exposed-patients analysis (v2)
The practice models pick **patients**, not single visits, so they match how the real models were trained. The slow part is training the practice models (about an hour in total). Finished ones are skipped, so run it in chunks if needed by changing `RUN`.

In [ ]:
import os, sys, json, gc, time, warnings, subprocess
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = os.environ.get("PA_ROOT", "/content/drive/MyDrive/privacy-audit/")
sys.path.insert(0, ROOT)          # pa_lib.py must sit in the privacy-audit folder on Drive
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip('opacus')
from pa_lib import *
make_dirs()
print(versions())


Mounted at /content/drive
{'python': '3.13.18', 'numpy': '2.1.3', 'pandas': '2.2.3', 'torch': '2.11.0+cpu', 'opacus': '1.6.0', 'art': None, 'sklearn': '1.6.1'}


In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
Xtr, ytr = load_split("target_train"); Xte, yte = load_split("target_test")
input_dim = Xtr.shape[1]; n_tr = len(Xtr)
pool_X = np.concatenate([Xtr, Xte]); pool_y = np.concatenate([ytr, yte])
pool_pid = np.concatenate([load_patients("target_train"), load_patients("target_test")])
N = len(pool_y); idx = np.arange(N)
EARLY = get_config()["early_epochs"]
CONFIGS = {"baseline_e50": ("plain", EPOCHS, 32), "baseline_early": ("plain", EARLY, 32),
           "dpsgd_eps1": ("dp", 1, 16), "dpsgd_eps10": ("dp", 10, 16), "dpsgd_eps100": ("dp", 100, 16)}
def phi_of(model):
    z = get_logits(model, pool_X)
    return z[idx, pool_y] - z[idx, 1 - pool_y]
print("pool rows:", N, "| patients:", len(np.unique(pool_pid)), "| early epochs:", EARLY)


pool rows: 9116 | patients: 6375 | early epochs: 6


In [ ]:
RUN = ["baseline_early", "baseline_e50", "dpsgd_eps1", "dpsgd_eps10", "dpsgd_eps100"]    # trim this list to run in chunks
t0 = time.time()
for cfg in RUN:
    kind, arg, K = CONFIGS[cfg]
    os.makedirs(LIRA + cfg, exist_ok=True)
    for k in range(K):
        path = f"{LIRA}{cfg}/shadow_{k}.npz"
        if os.path.exists(path):
            continue
        mask = patient_masks(pool_pid, 0.7, np.random.RandomState(k))          # 70% of PATIENTS train this practice model
        if kind == "plain":
            m = train_plain(pool_X[mask], pool_y[mask], arg, 500 + k)[0]
        else:
            m = train_dp(pool_X[mask], pool_y[mask], arg, 500 + k)[0]
        np.savez(path, phi=phi_of(m), mask=mask)
        del m; gc.collect()
        print(f"{cfg}: practice model {k + 1}/{K} ({(time.time() - t0) / 60:.1f} min)")
print("practice models ready")


baseline_early: practice model 1/32 (0.1 min)
baseline_early: practice model 2/32 (0.2 min)
baseline_early: practice model 3/32 (0.2 min)
baseline_early: practice model 4/32 (0.2 min)
baseline_early: practice model 5/32 (0.3 min)
baseline_early: practice model 6/32 (0.3 min)
baseline_early: practice model 7/32 (0.3 min)
baseline_early: practice model 8/32 (0.3 min)
baseline_early: practice model 9/32 (0.4 min)
baseline_early: practice model 10/32 (0.4 min)
baseline_early: practice model 11/32 (0.4 min)
baseline_early: practice model 12/32 (0.4 min)
baseline_early: practice model 13/32 (0.4 min)
baseline_early: practice model 14/32 (0.4 min)
baseline_early: practice model 15/32 (0.5 min)
baseline_early: practice model 16/32 (0.5 min)
baseline_early: practice model 17/32 (0.5 min)
baseline_early: practice model 18/32 (0.5 min)
baseline_early: practice model 19/32 (0.5 min)
baseline_early: practice model 20/32 (0.5 min)
baseline_early: practice model 21/32 (0.6 min)
baseline_early: practi

In [ ]:
# attack the real models with LiRA (members = target_train patients, non-members = target_test patients)
labels = np.r_[np.ones(n_tr), np.zeros(N - n_tr)]
out_csv = RESULTS + "lira_results.csv"
done = set(pd.read_csv(out_csv)["model"]) if os.path.exists(out_csv) else set()
rows, lira_cache = [], {}
for cfg in RUN:
    if cfg in done:
        continue
    K = CONFIGS[cfg][2]
    files = [np.load(f"{LIRA}{cfg}/shadow_{k}.npz") for k in range(K)]
    phis = np.stack([f["phi"] for f in files]); masks = np.stack([f["mask"] for f in files])
    for seed in SEEDS:
        s = lira_scores(phi_of(load_model(cfg, seed, input_dim)), phis, masks)
        rows.append({"model": cfg, "seed": seed, "auc": roc_auc_score(labels, s),
                     "tpr@1%fpr": tpr_at_fpr(labels, s, 0.01), "tpr@0.1%fpr": tpr_at_fpr(labels, s, 0.001)})
if rows:
    pd.DataFrame(rows).to_csv(out_csv, mode="a", header=not os.path.exists(out_csv), index=False)
lr = pd.read_csv(out_csv)
order = [c for c in CONFIGS if c in set(lr["model"])]
print(lr.groupby("model")[["auc", "tpr@1%fpr", "tpr@0.1%fpr"]].agg(["mean", "std"]).reindex(order).round(3).to_string())
print("chance: AUC 0.5 | TPR@1%FPR 0.01 | TPR@0.1%FPR 0.001")


                  auc        tpr@1%fpr        tpr@0.1%fpr       
                 mean    std      mean    std        mean    std
model                                                           
baseline_e50    0.639  0.008     0.074  0.004       0.028  0.006
baseline_early  0.534  0.008     0.025  0.001       0.008  0.002
dpsgd_eps1      0.502  0.003     0.011  0.003       0.002  0.000
dpsgd_eps10     0.509  0.006     0.014  0.001       0.003  0.001
dpsgd_eps100    0.514  0.005     0.013  0.002       0.004  0.001
chance: AUC 0.5 | TPR@1%FPR 0.01 | TPR@0.1%FPR 0.001


In [ ]:
# which training patients are most exposed in the overfit model?
cfg = "baseline_e50"; K = CONFIGS[cfg][2]
files = [np.load(f"{LIRA}{cfg}/shadow_{k}.npz") for k in range(K)]
phis = np.stack([f["phi"] for f in files]); masks = np.stack([f["mask"] for f in files])
per_seed = np.array([lira_scores(phi_of(load_model(cfg, s, input_dim)), phis, masks)[:n_tr] for s in SEEDS])
score = per_seed.mean(0)
c = spearmanr(per_seed.T).correlation
agree = float(c[np.triu_indices(len(SEEDS), 1)].mean())
top = np.zeros(n_tr, dtype=bool); top[np.argsort(-score)[:int(0.05 * n_tr)]] = True
print(f"agreement between models on who is exposed: {agree:.2f} | readmitted share: top 5% = {ytr[top].mean():.3f}, everyone = {ytr.mean():.3f}")

def feature_table(X, mask):
    t = pd.DataFrame({"feature": feature_names(), "top5_mean": X[mask].mean(0), "rest_mean": X[~mask].mean(0),
                      "std_diff": (X[mask].mean(0) - X[~mask].mean(0)) / (X.std(0) + 1e-9)})
    return t.reindex(t["std_diff"].abs().sort_values(ascending=False).index)
pd.DataFrame({"member_row": np.arange(n_tr), "label": ytr, "exposure_score": score, "exposed_top5": top}).to_csv(RESULTS + "exposure_baseline_e50.csv", index=False)
feature_table(Xtr, top).to_csv(RESULTS + "exposure_features_baseline_e50.csv", index=False)

pos = np.where(ytr == 1)[0]; tp = np.zeros(len(pos), dtype=bool); tp[np.argsort(-score[pos])[:int(0.2 * len(pos))]] = True
within = feature_table(Xtr[pos], tp); within.to_csv(RESULTS + "exposure_features_within_readmitted.csv", index=False)
json.dump({"agreement_between_models": agree, "readmitted_share_top5": float(ytr[top].mean()), "readmitted_share_all": float(ytr.mean())},
          open(RESULTS + "exposure_summary.json", "w"))
print("\nwithin readmitted patients only (top 20% most exposed vs the rest):")
print(within.head(10).round(3).to_string(index=False))


agreement between models on who is exposed: 0.49 | readmitted share: top 5% = 0.903, everyone = 0.112

within readmitted patients only (top 20% most exposed vs the rest):
            feature  top5_mean  rest_mean  std_diff
          change_No      0.301      0.569    -0.537
   number_emergency      0.908      0.034     0.534
   time_in_hospital      0.584      0.036     0.513
   number_inpatient      0.996      0.286     0.477
        age_[40-50)      0.241      0.093     0.449
  number_outpatient      0.400     -0.065     0.441
 max_glu_serum_>300      0.068      0.007     0.436
    pioglitazone_No      0.872      0.959    -0.370
pioglitazone_Steady      0.113      0.034     0.366
         insulin_Up      0.211      0.095     0.358
